# Notebook Setup & Imports

In [2]:
import sys
from pathlib import Path

# Add project root to PYTHONPATH
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import torch
import pandas as pd
import numpy as np

# Import Project Modules

In [3]:
from src.dataset import load_data, make_data_loader, make_windows
from src.model import StockMLP, StockCNN
from src.train import train
from src.evaluate import evaluate

# Configuration

In [4]:
DATA_PATH = PROJECT_ROOT / "data" / "train.csv"
BATCH_SIZE = 128
WINDOW_SIZE = 30
EPOCHS = 15
USE_CNN = True  # switch between MLP and CNN
NORMALIZE = True

# Load Dataset

In [5]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Please place train.csv inside data/ directory."
    )

df = load_data(DATA_PATH)
df.head()

,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,ticker_1,1962-01-02,0.0,0.265828,0.261788,0.261788,25600.0,0.0,0.0
19,ticker_10,1962-01-02,0.0,0.274973,0.271247,0.272737,192000.0,0.0,0.0
18,ticker_2,1962-01-02,0.0,0.210033,0.203061,0.208290,2648000.0,0.0,0.0
17,ticker_3,1962-01-02,0.0,0.141439,0.139528,0.139528,77440.0,0.0,0.0
16,ticker_4,1962-01-02,0.0,1.559642,1.549128,1.556138,40740.0,0.0,0.0


# Creating Sliding Windows

In [6]:
X, y = make_windows(df, window=WINDOW_SIZE)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive ratio:", y.mean())


X shape: (20998879, 30)
y shape: (20998879,)
Positive ratio: -0.00019843916430015146


# Train / Validation Split

In [7]:
split_idx = int(len(X) * 0.8)

X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print("Train samples:", len(X_train))
print("Validation samples:", len(X_val))

Train samples: 16799103
Validation samples: 4199776


# DataLoaders

In [8]:
from torch.utils.data import TensorDataset, DataLoader

if NORMALIZE:
    from src.features import normalize
    X_train = normalize(X_train)
    X_val = normalize(X_val)

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Initialize Model

In [9]:
if USE_CNN:
    model = StockCNN()
    print("Using CNN model")
else:
    model = StockMLP(input_dim=WINDOW_SIZE)
    print("Using MLP model")

model

Using CNN model


StockCNN(
  (features): Sequential(
    (0): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(64, 128, kernel_size=(7,), stride=(1,), padding=(3,))
    (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): AdaptiveAvgPool1d(output_size=1)
  )
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)

# Train Model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()
print("hello")
train(
    model=model,
    loader=train_loader,
    epochs=EPOCHS,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
)


hello
Training started...
Device for training: cuda
Started:
Epoch 1/15 - Loss: -360363924.0858
Started:
Epoch 2/15 - Loss: -4936010875.7038
Started:


# Evaluate on Validation Set

In [ ]:
model.to(device)

val_accuracy = evaluate(
    model,
    X_val,
    y_val,
    device=device
)

print(f"Validation Accuracy: {val_accuracy:.4f}")


# Save Trained Model

In [ ]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "stock_model_v1.pt"
torch.save(model.state_dict(), model_path)

print(f"Model saved to: {model_path}")

# Quick Sanity Prediction

In [ ]:
model.eval()

sample = torch.tensor(X_val[:5], dtype=torch.float32).to(device)
with torch.no_grad():
    logits = model(sample)
    probs = torch.sigmoid(logits)

print("Predicted probabilities:", probs.cpu().numpy())
print("True labels:", y_val[:5])